# Etapa 00: Setup y Contrato de Datos

**Entorno:** CPU | **Memoria:** Estándar | **Dependencias Clave:** `numpy`, `pandas`

## Objetivo de la Libreta
Establecer la configuración global, las rutas de almacenamiento persistente en Google Drive, y los esquemas de datos requeridos para la replicación estricta del modelo DESOM (Teimoorinia et al., 2021) aplicado a galaxias MaNGA DR15.

## Rigor Computacional y Científico
1. **Configuración Centralizada:** Un único archivo `config.json` dicta los hiperparámetros de todas las etapas posteriores.
2. **Tolerancia a Fallos:** Montaje robusto de Drive con reintentos y fallback.
3. **Validación Física:** Se genera y audita la malla espectral log-espaciada en reposo (3650–8950 Å, 4544 bins), comprobando rigurosamente que las líneas de emisión esperadas caigan en los índices documentados (±2 bins).

In [ ]:
import os
import sys
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import drive

def mount_drive_with_retry(mount_point='/content/drive', max_retries=3):
    """Monta Google Drive con reintentos para asegurar estabilidad en el pipeline."""
    print(f"Intentando montar Google Drive en {mount_point}...")
    for attempt in range(1, max_retries + 1):
        try:
            if not os.path.exists(mount_point):
                drive.mount(mount_point, force_remount=True)
            else:
                print("El directorio ya existe. Verificando acceso...")
                # Fallback check
                if not os.path.exists(os.path.join(mount_point, 'MyDrive')):
                    drive.mount(mount_point, force_remount=True)
            print("✅ Google Drive montado exitosamente.")
            return True
        except Exception as e:
            print(f"⚠️ Intento {attempt} fallido: {e}")
            if attempt < max_retries:
                time.sleep(5)
    
    raise RuntimeError("❌ Error crítico: No se pudo montar Google Drive tras varios reintentos.")

mount_drive_with_retry()

# Definición de Rutas Estructurales
ROOT = Path('/content/drive/MyDrive/GalaxyAE_colab')
DATA_DIR = ROOT / 'data'
MODELS_DIR = ROOT / 'models'
AUDIT_DIR = ROOT / 'audit'

for d in [DATA_DIR, MODELS_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print(f"\nEstructura de directorios lista en: {ROOT}")

In [ ]:
# Modo Smoke Test (Para validaciones rápidas, cambia a 3. Para producción, 100)
N_GALAXIES = 100 

# Diccionario maestro de configuración basado en el Paper
CONFIG = {
    "project_name": "MaNGA_DESOM_Replication",
    "run_mode": "smoke_test" if N_GALAXIES < 100 else "production",
    "n_galaxies_target": N_GALAXIES,
    "seed": 42, # Semilla fija para numpy, tensorflow y split
    "versions": {
        "tensorflow": "2.20.0",
        "tf_keras": "2.20.1",
        "python": sys.version.split()[0]
    },
    "grid": {
        "wave_min": 3650,
        "wave_max": 8950,
        "n_bins": 4544,
        "norm_window": [4050, 4150],
        "sky_lines_observed": [5577, 6300, 5895],
        "em_lines_expected_indices": [108, 1554, 1603, 1453, 2973, 2989]
    },
    "voronoi": {
        "target_sn": 20,
        "min_sn_mask": 2,
        "sn_cut_threshold": 10,
        "min_good_pixels": 3900,
        "covariance_in_binning": False # Se evaluará dinámicamente en Libreta 01, valor por defecto
    },
    "desom1": {
        "map_size": [15, 15],
        "latent_dim": 256,
        "gamma": 5e-4,
        "lr": 5e-5,
        "batch_size": 128,
        "Tmax": 15.0,
        "Tmin": 0.7,
        "iterations": 65000,
        "som_iterations": 50000,
        "training_mode": "B" # Modo 'B' justificado por la muestra reducida de 100 galaxias frente a >4000 originales
    },
    "desom2": {
        "map_size": [10, 10], # Reducido justificado por la N de galaxias (evita overfitting topológico)
        "latent_dim": 128,
        "gamma": 8e-6,
        "lr": 1e-4,
        "batch_size": 1024,
        "Tmax": 15.0,
        "Tmin": 0.5,
        "iterations": 40000,
        "som_iterations": 35000,
        "clip_percentile": 99.9
    }
}

config_path = ROOT / 'config.json'
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=4)
    
print(f"✅ Contrato de configuración escrito en {config_path}")

In [ ]:
# Validación Matemática de la Malla Espectral
print("Auditando malla espectral...")

wave_min = CONFIG['grid']['wave_min']
wave_max = CONFIG['grid']['wave_max']
n_bins = CONFIG['grid']['n_bins']
expected_indices = CONFIG['grid']['em_lines_expected_indices']

# Creación de la malla log-espaciada según los parámetros del paper
log_edges = np.geomspace(wave_min, wave_max, n_bins + 1)
wave_centers = np.sqrt(log_edges[:-1] * log_edges[1:])

# Las líneas de emisión típicas en MaNGA DR15 usadas por el modelo:
# [OII]~3727, [OIII]~4959, [OIII]~5007, H_beta~4861, H_alpha~6563, [NII]~6584
rest_em_lines = [3727.09, 4958.91, 5006.84, 4861.32, 6562.80, 6583.41]

assert len(wave_centers) == n_bins, "Error: Dimensiones de la malla incorrectas."

print(f"Malla generada: {n_bins} bins, desde {wave_centers[0]:.2f} Å hasta {wave_centers[-1]:.2f} Å")

deviations_count = 0
for i, line in enumerate(rest_em_lines):
    # Encontrar el índice más cercano en la malla log-espaciada
    idx = np.argmin(np.abs(wave_centers - line))
    expected_idx = expected_indices[i]
    diff = abs(idx - expected_idx)
    
    print(f"Línea {line} Å -> Índice Calculado: {idx} | Índice Esperado: {expected_idx} | Delta: {diff}")
    if diff > 2:
        deviations_count += 1
        print(f"❌ ERROR: La línea {line} Å excede la tolerancia (±2 bins).")

if deviations_count > 0:
    raise ValueError("La malla generada no cuadra matemáticamente con los índices EM_LINES requeridos por el paper.")
else:
    print("✅ Malla validada con éxito. Tolerancia respetada.")

In [ ]:
# Contrato de Esquemas de Datos (NPZ)
SCHEMA_CONTRACT = {
    "consolidated_dataset": {
        "X": "(N, 4544) float32 - Espectros normalizados",
        "wave_rest": "(4544,) float64 - Malla en reposo log-espaciada",
        "mangaID": "(N,) str - Identificador MaNGA",
        "plateifu": "(N,) str - Identificador Plate-IFU",
        "baxel_id": "(N,) int32 - ID del contenedor Voronoi",
        "redshift": "(N,) float32 - Corrimiento al rojo",
        "sn_corr": "(N,) float32 - S/N corregida por factor de Law",
        "split": "(N,) str - 'train', 'val', 'test' a nivel de espectro (modo spectra)"
    }
}

schema_path = ROOT / 'npz_schema.json'
with open(schema_path, 'w') as f:
    json.dump(SCHEMA_CONTRACT, f, indent=4)
print("✅ Esquemas de datos NPZ definidos.")

In [ ]:
import pprint

# Generación de Tabla de Referencia y Registro de Desviaciones
compliance_table = pd.DataFrame([
    {"Parámetro": "Muestra", "Paper": "DR15, 0<z<0.15, un cubo/mangaID", "Implementado": "DR15, 0<z<0.15, 100 galaxias", "Estado": "DESVIACIÓN (Submuestra 100)"},
    {"Parámetro": "Malla en reposo", "Paper": "3650–8950 Å, 4544 bins", "Implementado": "3650–8950 Å, 4544 bins", "Estado": "LITERAL"},
    {"Parámetro": "EM_LINES validación", "Paper": "[108,1554,1603,1453,2973,2989] ±2", "Implementado": "Mismos índices auditados con éxito", "Estado": "LITERAL"},
    {"Parámetro": "Voronoi target S/N", "Paper": "20", "Implementado": "20", "Estado": "LITERAL"},
    {"Parámetro": "DESOM-1 Config", "Paper": "Modo A (Literal)", "Implementado": "Modo B (Ajuste iteraciones/batch)", "Estado": "ASUNCIÓN (Falta de N para convergencia)"},
    {"Parámetro": "DESOM-2 Config", "Paper": "15x15, p99.9 clip", "Implementado": "10x10, p99.9 clip", "Estado": "ASUNCIÓN (Reducción de grilla por N=100)"},
    {"Parámetro": "Split de datos", "Paper": "modo 'spectra'", "Implementado": "modo 'spectra' + metadata por galaxia", "Estado": "LITERAL"}
])

print("--- Tabla de Trazabilidad Metodológica ---")
display(compliance_table)

deviations_md = "# Registro de Desviaciones y Asunciones Metodológicas\n\n"
for _, row in compliance_table.iterrows():
    if row["Estado"] != "LITERAL":
        deviations_md += f"* **{row['Parámetro']}**: {row['Estado']}. Paper requiere '{row['Paper']}', pero se implementó '{row['Implementado']}'.\n"

with open(AUDIT_DIR / 'deviations.md', 'w') as f:
    f.write(deviations_md)

print(f"\n✅ Archivo de trazabilidad escrito en {AUDIT_DIR / 'deviations.md'}")
print("Setup Completado. Listo para continuar con la Libreta 01.")